# Symbolic Regression for Dictionary Discovery: the Levitt abortion-crime application

We study the effect of the effective abortion rate $d$ on the crime rate $y$ controlling for a
high-dimensional dictionary of state covariates $x$ (Donohue and Levitt 2001; Belloni, Chernozhukov
and Hansen 2014). Partially linear model
$$y=\theta\,d+g(x)+\varepsilon,\qquad d=m(x)+v.$$

**Estimators.** *PDS-LASSO* (BCH double selection over the hand-built 284-term dictionary); *SR-PDS*
(augment the dictionary with PySR-discovered nonlinear basis terms, then double-select; in-sample SE);
*SR-PDS-CF* (the same, cross-fitted, so inference is valid under a data-dependent dictionary).

**Argument.** §1-§3 set up the data and methods. §4 reports the three estimators across all three
crimes. §5 is the core: we (5.1) reproduce BCH's Table 2, (5.2) strip the dictionary to primitives and
show the classical methods lose the hand-built nonlinear controls, (5.3) show SR-PDS recovers the
full-dictionary result from primitives alone, and (5.4) display exactly which nonlinear terms SR
discovers for the crime and abortion equations -- i.e. it rediscovers, systematically, what BCH
specified by hand.

## 1. Setup

In [1]:
import sys, re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GroupKFold, KFold
from collections import Counter
from IPython.display import display

np.set_printoptions(suppress=True)

_here = Path.cwd()
DATA_DIR = next((p for p in [_here, _here / "levitt", _here.parent / "levitt"]
                 if list(p.glob("levitt_*.csv"))), _here)
print("DATA_DIR:", DATA_DIR.resolve())

# full PySR config for single in-depth runs; lighter sweep config for the multi-crime loops
# ---- discovery controls ----
POLY_ONLY  = False   # True drops division: discovered basis is purely polynomial (closest to BCH)
MAX_DEGREE = 4       # cap on total polynomial degree of discovered terms (3 = more conservative)

PYSR_CONFIG = dict(niterations=40, binary_operators=["+", "-", "*", "/"],
                   unary_operators=["square", "log", "abs"], select_k_features=10, maxsize=16,
                   populations=15, model_selection="best", parallelism="serial",
                   deterministic=True, random_state=0, verbosity=0, progress=False,
                   nested_constraints={"square": {"square": 0}})
if POLY_ONLY:
    PYSR_CONFIG["binary_operators"] = ["+", "-", "*"]
PYSR_SWEEP = {**PYSR_CONFIG, "niterations": 20, "populations": 8}

DATA_DIR: /Users/przemekszkodon/Documents/University of Cambridge/Dissertation/repo/enriched_post_double_selection/levitt


## 2. Data and design

`D` = first difference, `y` = crime (outcome), `x` = abortion rate (treatment); the final letter is the
crime type (m/v/p). Raw $y,d$ live in `linear`; the 284-term control dictionary in `controls_<crime>`.

In [2]:
data = {f.stem.replace("levitt_", ""): pd.read_csv(f) for f in sorted(DATA_DIR.glob("levitt_*.csv"))}
for k, v in data.items():
    print(f"{k:16s} {v.shape}  {list(v.columns)[:7]}{' ...' if v.shape[1] > 7 else ''}")

_YD = {"murd": ("Dym", "Dxm"), "viol": ("Dyv", "Dxv"), "prop": ("Dyp", "Dxp")}
CRIMES = ["viol", "prop", "murd"]

def load_crime(crime):
    """Return y, d, X, feature_names, state for one crime."""
    yc, dc = _YD[crime]
    y = data["linear"][yc].to_numpy(float)
    d = data["linear"][dc].to_numpy(float)
    Xc = data[f"controls_{crime}"].select_dtypes("number").dropna(axis=1)
    names, X = list(Xc.columns), Xc.to_numpy(float)
    m = min(len(y), len(d), len(X))
    state = data["linear"]["state"].to_numpy()[:m]
    return y[:m], d[:m], X[:m], names, state

control_names    (284, 4)  ['index', 'viol', 'prop', 'murd']
controls_murd    (576, 284)  ['Dprison', 'Dpolice', 'Dur', 'Dinc', 'Dpov', 'Dafdc', 'Dbeer'] ...
controls_prop    (576, 284)  ['Dprison', 'Dpolice', 'Dur', 'Dinc', 'Dpov', 'Dafdc', 'Dbeer'] ...
controls_viol    (576, 284)  ['Dprison', 'Dpolice', 'Dur', 'Dinc', 'Dpov', 'Dafdc', 'Dbeer'] ...
linear           (576, 7)  ['state', 'Dyv', 'Dxv', 'Dyp', 'Dxp', 'Dym', 'Dxm']
partialled       (576, 7)  ['state', 'DxV', 'DyV', 'DxP', 'DyP', 'DxM', 'DyM']
state            (576, 1)  ['state']


## 3. Methods

### 3.1 Utilities

In [3]:
def zscore(c): return (c - c.mean()) / (c.std() + 1e-12)

def resid_frac(z, KZ):
    if KZ.shape[1] == 0: return 1.0
    b, *_ = np.linalg.lstsq(KZ, z, rcond=None); r = z - KZ @ b
    return float(r @ r) / float(z @ z)

def sane_col(c, max_abs=1e6):
    c = np.asarray(c, float)
    return c.ndim == 1 and np.all(np.isfinite(c)) and np.std(c) > 0 and np.max(np.abs(c)) < max_abs

def sanitize_names(names):
    safe, used = [], set()
    for nm in names:
        s = re.sub(r"[^0-9A-Za-z_]", "_", str(nm))
        if not s or s[0].isdigit(): s = "v_" + s
        base, k = s, 1
        while s in used: s = f"{base}_{k}"; k += 1
        used.add(s); safe.append(s)
    return safe

def round_consts(expr, n=3):
    import sympy
    return expr.xreplace({c: round(float(c), n) for c in expr.atoms(sympy.Float)})

_STEM = {"prison": "prison pop.", "police": "police", "ur": "unemployment", "inc": "income",
         "pov": "poverty", "afdc": "AFDC", "beer": "beer", "gun": "gun law",
         "xM": "abortion(murd)", "xV": "abortion(viol)", "xP": "abortion(prop)"}
def _gloss(tok):
    if tok == "t": return "trend"
    if tok == "Abs": return "abs"
    pre = ""
    if tok[:1] == "D": pre = "\u0394 "; tok = tok[1:]
    elif tok[:1] == "L": pre = "lag "; tok = tok[1:]
    suf = ""
    if tok.endswith("Bar"): suf = " (mean)"; tok = tok[:-3]
    if tok.endswith("0"): suf = " (init)" + suf; tok = tok[:-1]
    return f"{pre}{_STEM.get(tok, tok)}{suf}".strip()
def prettify(name): return re.sub(r"[A-Za-z]+[0-9]*", lambda m: _gloss(m.group(0)), str(name))

### 3.2 PDS-LASSO (homoskedastic plug-in penalty, HC1 SE)

In [4]:
def _plugin_alpha(n, p, sigma, c=1.1, gamma=None):
    if gamma is None: gamma = 0.1 / np.log(max(n, 3))
    return c * sigma * np.sqrt(2.0 * np.log(2.0 * p / gamma) / n)

def rlasso_select(X, y, c=1.1, gamma=None, n_iter=2, tol=1e-8):
    n, p = X.shape
    mu, sd = X.mean(0), X.std(0); sd[sd == 0] = 1.0; Xs = (X - mu) / sd
    sigma = np.std(y - y.mean()); sel = np.array([], int)
    for _ in range(n_iter):
        a = _plugin_alpha(n, p, sigma, c, gamma)
        m = Lasso(alpha=a, fit_intercept=True, max_iter=20000).fit(Xs, y)
        sel = np.where(np.abs(m.coef_) > tol)[0]
        if 0 < len(sel) < n - 1:
            r = y - LinearRegression().fit(Xs[:, sel], y).predict(Xs[:, sel])
            sigma = np.sqrt(r @ r / max(n - len(sel), 1))
        else: sigma = np.std(y - y.mean())
    return sel

def _ols_hc1(y, Z):
    n, k = Z.shape; XtXi = np.linalg.pinv(Z.T @ Z); b = XtXi @ Z.T @ y; r = y - Z @ b
    V = XtXi @ (Z.T @ (r[:, None] ** 2 * Z)) @ XtXi * (n / max(n - k, 1)); return b, V

def pds_lasso(y, d, X, **kw):
    y, d, X = np.asarray(y, float).ravel(), np.asarray(d, float).ravel(), np.asarray(X, float)
    n = len(y); s_y = rlasso_select(X, y, **kw); s_d = rlasso_select(X, d, **kw)
    sel = np.union1d(s_y, s_d).astype(int)
    Z = np.column_stack([np.ones(n), d, X[:, sel]]) if len(sel) else np.column_stack([np.ones(n), d])
    b, V = _ols_hc1(y, Z); se = float(np.sqrt(V[1, 1]))
    return dict(theta=float(b[1]), se=se, t=float(b[1] / se), n_selected=len(sel),
                selected=sel, s_outcome=s_y, s_treatment=s_d)

### 3.3 Symbolic discovery (shared)

`discover_terms` returns each basis term as a frozen callable (so it can be evaluated on held-out
folds). `dedup_terms` keeps only genuinely new columns -- not collinear with an existing one, not
reconstructable from terms already kept.

In [5]:
def _within_degree(core, max_degree):
    """Reject overfitting high-order terms: any explicit power > cap, or total degree > cap."""
    import sympy
    for pw in core.atoms(sympy.Pow):
        if pw.exp.is_number and abs(float(pw.exp)) > max_degree:
            return False
    try:
        if sympy.Poly(core, *core.free_symbols).total_degree() > max_degree:
            return False
    except Exception:
        pass        # non-polynomial (abs/log/ratio): rely on the explicit-power check
    return True

def discover_terms(X, target, names, config, base_features=None, max_terms=24, max_ops=8, max_degree=None):
    from pysr import PySRRegressor
    import sympy
    if max_degree is None: max_degree = globals().get("MAX_DEGREE", 4)
    idx = ([names.index(f) for f in base_features] if base_features is not None else list(range(X.shape[1])))
    safe = sanitize_names([names[i] for i in idx])
    model = PySRRegressor(**config); model.fit(X[:, idx], np.asarray(target, float).ravel(), variable_names=safe)
    syms = sympy.symbols(safe)
    if not isinstance(syms, (list, tuple)): syms = (syms,)
    relabel = {syms[i]: sympy.Symbol(str(names[idx[i]])) for i in range(len(syms))}
    terms, seen = [], set()
    for expr in model.equations_["sympy_format"]:
        for term in sympy.Add.make_args(sympy.expand(expr)):
            if term.is_number: continue
            _, core = term.as_coeff_Mul()
            if sympy.count_ops(core) > max_ops: continue
            if not _within_degree(core, max_degree): continue   # drop overfitting high powers
            key = str(core)
            if key in seen: continue
            seen.add(key)
            try:
                f = sympy.lambdify(syms, core, "numpy")
                terms.append((tuple(idx), f, str(round_consts(core).xreplace(relabel))))
            except Exception: continue
            if len(terms) >= max_terms: break
        if len(terms) >= max_terms: break
    return terms

def eval_term(term, X):
    idx, f, _ = term; return np.asarray(f(*[X[:, i] for i in idx]), float)

def dedup_terms(term_lists, Xref, corr_existing=0.99, r2_among=0.95):
    Xz = np.column_stack([zscore(Xref[:, j]) for j in range(Xref.shape[1])])
    kept, kz = [], []
    for terms in term_lists:
        for t in terms:
            col = eval_term(t, Xref)
            if not sane_col(col): continue
            z = zscore(col)
            if np.max(np.abs(Xz.T @ z) / len(z)) > corr_existing: continue
            KZ = np.column_stack(kz) if kz else np.empty((len(z), 0))
            if resid_frac(z, KZ) < (1 - r2_among): continue
            kept.append(t); kz.append(z)
    return kept

def build_aug(X, terms):
    if not terms: return X
    return np.column_stack([X] + [np.nan_to_num(eval_term(t, X), nan=0., posinf=0., neginf=0.) for t in terms])

### 3.4 SR-PDS (in-sample) and SR-PDS-CF (cross-fitted)

In [6]:
def sr_pds(y, d, X, names, config, base_features=None, **kw):
    X = np.asarray(X, float)
    kept = dedup_terms([discover_terms(X, y, names, config, base_features),
                        discover_terms(X, d, names, config, base_features)], X)
    Xa = build_aug(X, kept); na = list(names) + [t[2] for t in kept]
    r = pds_lasso(y, d, Xa, **kw)
    r.update(discovered=[t[2] for t in kept], n_discovered=len(kept), n_dictionary=Xa.shape[1], names_aug=na)
    return r

def _post_lasso_fit(Z, target, sel):
    n = len(target); A = np.column_stack([np.ones(n), Z[:, sel]]) if len(sel) else np.ones((n, 1))
    beta = np.linalg.pinv(A.T @ A) @ A.T @ target
    return lambda Zn: (np.column_stack([np.ones(Zn.shape[0]), Zn[:, sel]]) if len(sel)
                       else np.ones((Zn.shape[0], 1))) @ beta

def sr_pds_cf(y, d, X, names, config, n_folds=5, groups=None, base_features=None,
              leak_corr=0.995, random_state=0, **kw):
    y, d, X = np.asarray(y, float).ravel(), np.asarray(d, float).ravel(), np.asarray(X, float); n = len(y)
    split = (GroupKFold(n_folds).split(X, y, groups) if groups is not None
             else KFold(n_folds, shuffle=True, random_state=random_state).split(X))
    yr, vr = np.empty(n), np.empty(n); disc, sel_u, sup = set(), set(), Counter()
    for tr, te in split:
        Xtr, Xte = X[tr], X[te]
        kept = dedup_terms([discover_terms(Xtr, y[tr], names, config, base_features),
                            discover_terms(Xtr, d[tr], names, config, base_features)], Xtr)
        na = list(names) + [t[2] for t in kept]; disc.update(na[len(names):]); sup.update(na[len(names):])
        Ztr, Zte = build_aug(Xtr, kept), build_aug(Xte, kept)
        ok = [j for j in range(Ztr.shape[1])                       # out-of-sample stability guard
              if sane_col(Ztr[:, j]) and np.all(np.isfinite(Zte[:, j]))
              and np.max(np.abs(Zte[:, j])) <= 100 * max(np.max(np.abs(Ztr[:, j])), 1e-8)]
        Ztr, Zte, na = Ztr[:, ok], Zte[:, ok], [na[j] for j in ok]
        dz = zscore(d[tr])                                          # treatment-leak guard
        keep_c = [j for j in range(Ztr.shape[1]) if abs(float(zscore(Ztr[:, j]) @ dz) / len(dz)) <= leak_corr]
        Ztr, Zte, na = Ztr[:, keep_c], Zte[:, keep_c], [na[j] for j in keep_c]
        s = np.union1d(rlasso_select(Ztr, y[tr], **kw), rlasso_select(Ztr, d[tr], **kw)).astype(int)
        sel_u.update(na[i] for i in s)
        yr[te] = y[te] - _post_lasso_fit(Ztr, y[tr], s)(Zte)
        vr[te] = d[te] - _post_lasso_fit(Ztr, d[tr], s)(Zte)
    vd = float(np.var(d)); r2 = float(1 - np.var(vr) / vd) if vd > 0 else 1.0
    J = float(np.mean(vr ** 2)); theta = float(np.sum(vr * yr) / np.sum(vr ** 2)); psi = vr * (yr - theta * vr)
    meat = (float(sum(psi[np.asarray(groups) == g].sum() ** 2 for g in np.unique(groups)))
            if groups is not None else float(np.sum(psi ** 2)))
    se = float(np.sqrt(meat / (n ** 2 * J ** 2)))
    return dict(theta=theta, se=se, t=theta / se, r2_treatment=r2, leak_warning=r2 > 0.99,
                n_discovered=len(disc), discovered_union=sorted(disc), fold_support=dict(sup))

### 3.5 BCH replication machinery and the stripped dictionary

`make_time_dummies` recovers BCH's trend from a $(v, v{\cdot}t)$ pair and builds period dummies;
`pds_lasso_bch` forces them in unpenalized and clusters SEs by state, using the cluster-robust rigorous
penalty (`rlasso_select_cr`). `build_stripped` reduces the dictionary to primitives (main effects, the
trend, and reconstructed state means) so SR must rediscover the nonlinear structure.

In [7]:
def make_time_dummies(names, X, state):
    cols = {nm: i for i, nm in enumerate(names)}
    pairs = [(cols[nm[:-2]], i) for nm, i in cols.items() if nm.endswith("*t") and nm[:-2] in cols]
    n = X.shape[0]
    if pairs:
        E = np.full((n, len(pairs)), np.nan)
        for k, (vi, ti) in enumerate(pairs):
            v = X[:, vi]; m = np.abs(v) > 1e-6; E[m, k] = X[m, ti] / v[m]
        t = np.nanmedian(E, axis=1); uq = np.unique(np.round(t[np.isfinite(t)], 4))
        period = np.array([int(np.argmin(np.abs(uq - round(float(x), 4)))) if np.isfinite(x) else 0 for x in t])
        src = "recovered BCH trend"
    else:
        period = np.zeros(n, int)
        for s in np.unique(state): m = state == s; period[m] = np.arange(m.sum())
        src = "within-state index"
    P = int(period.max()) + 1; D = np.zeros((n, P)); D[np.arange(n), period] = 1.0
    return D, src, P

def _ols_cluster(y, Z, groups):
    n, k = Z.shape; XtXi = np.linalg.pinv(Z.T @ Z); b = XtXi @ Z.T @ y; r = y - Z @ b
    gs = np.unique(groups); meat = np.zeros((k, k))
    for g in gs: s = Z[groups == g].T @ r[groups == g]; meat += np.outer(s, s)
    adj = (len(gs) / (len(gs) - 1)) * ((n - 1) / (n - k)); return b, XtXi @ meat @ XtXi * adj

def _partial_out(M, G): return M - G @ np.linalg.lstsq(G, M, rcond=None)[0]
def _coef_se(y, Z, groups, j): b, V = _ols_cluster(y, Z, groups); return float(b[j]), float(np.sqrt(V[j, j]))

def rlasso_select_cr(X, y, groups, c=1.1, gamma=None, n_iter=15):
    n, p = X.shape
    if gamma is None: gamma = 0.1 / np.log(n)
    gs = np.unique(groups); mu, sd = X.mean(0), X.std(0); sd[sd == 0] = 1.0; Xs = (X - mu) / sd
    alpha = c * norm.ppf(1 - gamma / (2 * p)) / np.sqrt(n); resid = y - y.mean(); sel = np.array([], int)
    for _ in range(n_iter):
        psi = np.zeros(p)
        for g in gs: psi += (Xs[groups == g].T @ resid[groups == g]) ** 2
        psi = np.sqrt(psi / n); psi[psi < 1e-8] = 1e-8
        m = Lasso(alpha=alpha, fit_intercept=True, max_iter=50000).fit(Xs / psi, y)
        sn = np.where(np.abs(m.coef_) > 1e-10)[0]
        resid = (y - LinearRegression().fit(Xs[:, sn], y).predict(Xs[:, sn])) if 0 < len(sn) < n - 1 else y - y.mean()
        if set(sn.tolist()) == set(sel.tolist()): sel = sn; break
        sel = sn
    return sel

def first_difference(y, d, G, groups): return _coef_se(y, np.column_stack([G, d]), groups, G.shape[1])
def all_controls(y, d, X, G, groups): return _coef_se(y, np.column_stack([G, d, X]), groups, G.shape[1])

def pds_lasso_bch(y, d, X, G, groups, cr_penalty=False, **kw):
    yt, dt, Xt = _partial_out(y, G), _partial_out(d, G), _partial_out(X, G)
    if cr_penalty: s_y, s_d = rlasso_select_cr(Xt, yt, groups), rlasso_select_cr(Xt, dt, groups)
    else: s_y, s_d = rlasso_select(Xt, yt, **kw), rlasso_select(Xt, dt, **kw)
    sel = np.union1d(s_y, s_d).astype(int)
    Z = np.column_stack([G, d, X[:, sel]]) if len(sel) else np.column_stack([G, d])
    th, se = _coef_se(y, Z, groups, G.shape[1]); return th, se, len(sel)

def is_constructed(name): return ("*" in str(name)) or ("^" in str(name))

def build_stripped(names, X, state):
    """Reduce the dictionary to primitives: main effects + recovered trend t + state means
    (reconstructed if they only appear interacted). Never includes products or powers."""
    cols = {nm: i for i, nm in enumerate(names)}
    base = {nm: X[:, i] for nm, i in cols.items() if not is_constructed(nm)}
    t, exact = None, False
    for nm, i in cols.items():
        if nm.endswith("*t") and nm[:-2] in cols:
            v = X[:, cols[nm[:-2]]]; m = np.abs(v) > 1e-6
            if m.mean() > 0.5: t = np.zeros(len(v)); t[m] = X[m, i] / v[m]; exact = True; break
    if t is None:
        t = np.zeros(len(X))
        for s in np.unique(state): m = state == s; t[m] = np.arange(m.sum())
    base["t"] = t
    if exact:
        tm = np.abs(t) > 1e-6
        for nm, i in list(cols.items()):
            for suf, den in (("*t", t), ("*t^2", t ** 2)):
                if nm.endswith(suf):
                    fac = nm[:-len(suf)]
                    if fac and fac not in base and not is_constructed(fac):
                        f = np.zeros(len(X)); f[tm] = X[tm, i] / den[tm]
                        if np.all(np.isfinite(f)) and np.std(f) > 0: base[fac] = f
                    break
    bn = list(base); return np.column_stack([base[k] for k in bn]), bn

def sr_discoveries(y, d, X, names, config, base_features=None):
    """Nonlinear terms SR discovers, separately for the outcome (y) and treatment (d) equations."""
    ky = dedup_terms([discover_terms(X, y, names, config, base_features)], X)
    kd = dedup_terms([discover_terms(X, d, names, config, base_features)], X)
    return [t[2] for t in ky], [t[2] for t in kd]

## 4. Main results: three estimators across all three crimes

The full hand-built dictionary. `RUN_HEAVY` controls the PySR-based rows (slow); set it `False` for a
quick PDS-LASSO-only pass.

In [8]:
RUN_HEAVY = True
main = {}
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    rec = {"pds": pds_lasso(y, d, X)}
    if RUN_HEAVY:
        rec["sr"] = sr_pds(y, d, X, names, PYSR_SWEEP)
        rec["cf"] = sr_pds_cf(y, d, X, names, PYSR_SWEEP, n_folds=5, groups=state)
    main[cr] = rec
print("done:", list(main))

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Using features ['Lbeer' 'Lbeer_2' 'Dur_Dinc' 'Dur_Dpov_t' 'Dprison_Dpov_t_2' 'Dpolice0'
 'Dprison0_2' 'Dpolice0_2' 'Dpolice0_t' 'urBar_2_t']
Using features ['Lafdc' 'Lafdc_2' 'Lpolice0_t_2' 'Linc0_t_2' 'DxV0' 'DxV0_2' 'DxV0_t'
 'DxV0_t_2' 'xV0' 'xV0_t']
Using features ['Lprison' 'Lbeer' 'Lbeer_2' 'Dprison_Dafdc' 'Dur_Dinc' 'Dur_Dafdc'
 'Dinc_Dafdc_t_2' 'Dpolice0' 'Dpolice0_2' 'Dpov0_2_t']
Using features ['Lpolice0' 'Lpolice0_2' 'Dprison0_2_t' 'Lbeer0_2_t_2' 'policeBar'
 'policeBar_2' 'DxV0' 'DxV0_2' 'DxV0_t_2' 'xV0']
Using features ['Lbeer' 'Lbeer_2' 'Dbeer_t' 'Dur_Dinc' 'Dprison_Dpov_t'
 'Dprison_Dinc_t_2' 'Dprison_Dpov_t_2' 'Dprison0' 'Dpolice0' 'Dprison0_2']
Using features ['Lafdc' 'Lafdc_2' 'Linc0_t_2' 'Linc0_2_t_2' 'DxV0' 'DxV0_2' 'DxV0_t'
 'DxV0_t_2' 'xV0' 'xV0_t']
Using features ['Lpolice' 'Dur_Dinc' 'Dprison_Dpov_t' 'Dur_Dinc_t' 'Dprison_Dpov_t_2'
 'Dinc_Dafdc_t_2' 'Dpov_Dafdc_t_2' 'Dpolice0' 'urBar_t' 'prisonBar_t_2']
Using features ['Lafdc_2' 'Lpov0_t' 'povBar_t' 'DxV0' 'DxV0

done: ['viol', 'prop', 'murd']


In [9]:
rows = []
for cr in CRIMES:
    R = main[cr]
    rows.append(dict(crime=cr, method="PDS-LASSO", theta=R["pds"]["theta"], se=R["pds"]["se"], t=R["pds"]["t"]))
    if "sr" in R:
        rows.append(dict(crime=cr, method="SR-PDS", theta=R["sr"]["theta"], se=R["sr"]["se"], t=R["sr"]["t"]))
        rows.append(dict(crime=cr, method="SR-PDS-CF", theta=R["cf"]["theta"], se=R["cf"]["se"], t=R["cf"]["t"]))
_INFER = {"PDS-LASSO": "HC1 (valid)", "SR-PDS": "in-sample (descriptive only)",
          "SR-PDS-CF": "cross-fit + clustered (valid)"}
tab = pd.DataFrame(rows); tab["inference"] = tab["method"].map(_INFER)
display(tab.set_index(["crime", "method"]).round(4))
if RUN_HEAVY:
    print("\nCF treatment-reconstruction check R^2(d|x):",
          {cr: round(main[cr]["cf"]["r2_treatment"], 3) for cr in CRIMES})

theta      se       t                      inference
crime method                                                          
viol  PDS-LASSO -0.1457  0.1098 -1.3264                    HC1 (valid)
      SR-PDS    -0.1457  0.1098 -1.3264   in-sample (descriptive only)
      SR-PDS-CF -0.1709  0.1144 -1.4940  cross-fit + clustered (valid)
prop  PDS-LASSO -0.0527  0.0406 -1.2977                    HC1 (valid)
      SR-PDS    -0.0857  0.0481 -1.7826   in-sample (descriptive only)
      SR-PDS-CF -0.1015  0.0512 -1.9818  cross-fit + clustered (valid)
murd  PDS-LASSO -0.1684  0.4201 -0.4009                    HC1 (valid)
      SR-PDS    -0.1675  0.4204 -0.3985   in-sample (descriptive only)
      SR-PDS-CF -0.1795  0.1734 -1.0349  cross-fit + clustered (valid)


CF treatment-reconstruction check R^2(d|x): {'viol': 0.729, 'prop': 0.556, 'murd': 0.737}


## 5. Reproducing BCH and recovering the dictionary

### 5.1 Reproducing BCH (2014) Table 2

Their exact specification: time dummies forced in unpenalized, state-clustered SEs, cluster-robust
rigorous penalty for selection. We reproduce the three estimable rows for all three crimes.

In [10]:
BCH_T2 = {
    "First-difference":      {"viol": (-0.152, 0.034), "prop": (-0.108, 0.022), "murd": (-0.204, 0.068)},
    "All controls":          {"viol": ( 0.014, 0.719), "prop": (-0.195, 0.225), "murd": ( 2.343, 2.798)},
    "Post-double-selection": {"viol": (-0.104, 0.107), "prop": (-0.030, 0.055), "murd": (-0.125, 0.151)},
}
rows = []
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    D, src, P = make_time_dummies(names, X, state)
    G = np.column_stack([np.ones(len(y)), D[:, 1:]])
    fd, ac = first_difference(y, d, G, state), all_controls(y, d, X, G, state)
    th, se, _ = pds_lasso_bch(y, d, X, G, state)
    for spec, est in [("First-difference", fd), ("All controls", ac), ("Post-double-selection", (th, se))]:
        bt, bse = BCH_T2[spec][cr]
        rows.append({"spec": spec, "crime": cr, "theta": round(est[0], 3), "se": round(est[1], 3),
                     "BCH theta": bt, "BCH se": bse})
print(f"time dummies: {P} periods ({src})")
display(pd.DataFrame(rows).set_index(["spec", "crime"]))

time dummies: 12 periods (recovered BCH trend)


,,theta,se,BCH theta,BCH se
spec,crime,,,,
First-difference,viol,-0.152,0.034,-0.152,0.034
All controls,viol,0.126,0.703,0.014,0.719
Post-double-selection,viol,-0.146,0.112,-0.104,0.107
First-difference,prop,-0.108,0.022,-0.108,0.022
All controls,prop,-0.283,0.212,-0.195,0.225
Post-double-selection,prop,-0.053,0.055,-0.030,0.055
First-difference,murd,-0.204,0.067,-0.204,0.068
All controls,murd,1.357,1.747,2.343,2.798
Post-double-selection,murd,-0.168,0.197,-0.125,0.151


### 5.2 The role of the hand-built dictionary: full vs stripped

BCH's nonlinear basis is doing work. We strip the dictionary to primitives (main effects + trend +
means; no products or powers) and re-run classical PDS-LASSO. The estimate shifts and the
trend-interaction controls can no longer be selected, because they no longer exist.

In [11]:
rows = []
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    Xs, sn = build_stripped(names, X, state)
    full, strip = pds_lasso(y, d, X), pds_lasso(y, d, Xs)
    rows.append({"crime": cr, "full theta": round(full["theta"], 4), "full se": round(full["se"], 4),
                 "full p": X.shape[1], "stripped theta": round(strip["theta"], 4),
                 "stripped se": round(strip["se"], 4), "stripped p": Xs.shape[1]})
display(pd.DataFrame(rows).set_index("crime"))

,full theta,full se,full p,stripped theta,stripped se,stripped p
crime,,,,,,
viol,-0.1457,0.1098,284,-0.1726,0.0968,39
prop,-0.0527,0.0406,284,-0.0629,0.0431,39
murd,-0.1684,0.4201,284,-0.2952,0.2945,39


### 5.3 SR-PDS recovers the result from primitives

Now SR searches the stripped dictionary, discovers nonlinear terms, and double-selects. If the
main-effects-plus-SR estimate tracks the full hand-built dictionary, SR has automated the construction
of the nonlinear basis.

*Read the `recovers (<=1 SE)` column as the headline of this table:* does the primitives+SR estimate land within one standard error of the hand-built-dictionary estimate? Estimate-equivalence is the claim; term-counting (5.4) is supporting detail.

In [12]:
rows = []
strip_runs = {}
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    Xs, sn = build_stripped(names, X, state)
    full = main[cr]["pds"]
    srp = sr_pds(y, d, Xs, sn, PYSR_SWEEP)
    strip_runs[cr] = srp
    rows.append({"crime": cr, "BCH dict theta": round(full["theta"], 4), "BCH dict se": round(full["se"], 4),
                 "primitives+SR theta": round(srp["theta"], 4), "primitives+SR se": round(srp["se"], 4),
                 "dtheta vs BCH": round(srp["theta"] - full["theta"], 4),
                 "recovers (<=1 SE)": "yes" if abs(srp["theta"] - full["theta"]) <= full["se"] else "NO",
                 "SR terms added": srp["n_discovered"]})
display(pd.DataFrame(rows).set_index("crime"))

Using features ['Dprison' 'Dur' 'Dinc' 'Dbeer' 'Lpolice' 'Lur' 'Linc' 'Lbeer' 'Dprison0'
 'Dpolice0']
Using features ['Lpolice' 'Lur' 'Linc' 'Lafdc' 'Lbeer' 'Linc0' 'afdcBar' 'DxV0' 'xV0' 't']
Using features ['Dprison' 'Dinc' 'Lpolice' 'Lur' 'Lpov' 'Lafdc' 'Lbeer' 'Dpolice0'
 'Dafdc0' 'Linc0']
Using features ['Dbeer' 'Lpolice' 'Lur' 'Lafdc' 'Dafdc0' 'Linc0' 'afdcBar' 'DxP0' 'xP0'
 't']
Using features ['Dprison' 'Dpolice' 'Dur' 'Dinc' 'Dpov' 'Dafdc' 'Lprison' 'Lpolice' 'Lur'
 'prisonBar']
Using features ['Lprison' 'Lpolice' 'Linc' 'Lpov' 'Lbeer' 'povBar' 'afdcBar' 'DxM0' 'xM0'
 't']


,BCH dict theta,BCH dict se,primitives+SR theta,primitives+SR se,dtheta vs BCH,recovers (<=1 SE),SR terms added
crime,,,,,,,
viol,-0.1457,0.1098,-0.1337,0.1056,0.0120,yes,5
prop,-0.0527,0.0406,-0.1225,0.0465,-0.0698,NO,6
murd,-0.1684,0.4201,-0.1697,0.3776,-0.0013,yes,6


### 5.4 What SR discovers: nonlinear terms by equation

For each crime we list the nonlinear terms SR produced, split by the equation that produced them: the
**outcome (crime)** equation $y\sim x$ and the **treatment (abortion)** equation $d\sim x$. Terms in the
treatment equation are the potential confounders (they predict abortion); terms in the outcome equation
sharpen the crime model.

In [13]:
discovery = {}
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    Xs, sn = build_stripped(names, X, state)
    discovery[cr] = sr_discoveries(y, d, Xs, sn, PYSR_SWEEP)

for cr in CRIMES:
    ye, de = discovery[cr]
    allt = list(dict.fromkeys(ye + de))
    tbl = pd.DataFrame([{"nonlinear term": prettify(t),
                         "outcome eq (crime)": "x" if t in ye else "",
                         "treatment eq (abortion)": "x" if t in de else ""} for t in allt])
    print(f"\n===== {cr}: SR-discovered nonlinear terms =====")
    display(tbl if len(tbl) else "(none)")

Using features ['Dprison' 'Dur' 'Dinc' 'Dbeer' 'Lpolice' 'Lur' 'Linc' 'Lbeer' 'Dprison0'
 'Dpolice0']
Using features ['Lpolice' 'Lur' 'Linc' 'Lafdc' 'Lbeer' 'Linc0' 'afdcBar' 'DxV0' 'xV0' 't']
Using features ['Dprison' 'Dinc' 'Lpolice' 'Lur' 'Lpov' 'Lafdc' 'Lbeer' 'Dpolice0'
 'Dafdc0' 'Linc0']
Using features ['Dbeer' 'Lpolice' 'Lur' 'Lafdc' 'Dafdc0' 'Linc0' 'afdcBar' 'DxP0' 'xP0'
 't']
Using features ['Dprison' 'Dpolice' 'Dur' 'Dinc' 'Dpov' 'Dafdc' 'Lprison' 'Lpolice' 'Lur'
 'prisonBar']
Using features ['Lprison' 'Lpolice' 'Linc' 'Lpov' 'Lbeer' 'povBar' 'afdcBar' 'DxM0' 'xM0'
 't']



===== viol: SR-discovered nonlinear terms =====


,nonlinear term,outcome eq (crime),treatment eq (abortion)
0,Δ beer**2,x,
1,Δ income/(lag beer - 0.097),x,
2,abs(Δ abortion(viol) (init)),,x
3,lag income*trend,,x
4,lag income (init)*trend,,x



===== prop: SR-discovered nonlinear terms =====


,nonlinear term,outcome eq (crime),treatment eq (abortion)
0,lag poverty**2,x,
1,lag AFDC*lag income (init),x,
2,lag income (init)*abs(lag police),x,
3,Δ abortion(prop) (init)**2,,x
4,trend*abortion(prop) (init)**2,,x
5,abs(trend*abortion(prop) (init)),,x



===== murd: SR-discovered nonlinear terms =====


,nonlinear term,outcome eq (crime),treatment eq (abortion)
0,Δ AFDC**2,x,
1,abs(Δ police),x,
2,Δ income/Δ AFDC,x,
3,Δ income**2/Δ AFDC**2,x,
4,Δ unemployment/(-0.75*lag prison pop. - 0.981),x,
5,lag income*trend,,x


### 5.5 Headline estimates (report these)

The estimates to put in the dissertation: BCH's published post-double-selection effect, our PDS-LASSO
(BCH specification: time dummies, state-clustered SE), and our SR-PDS-CF (cross-fitted, clustered).
All three use valid inference. The in-sample SR-PDS standard errors in $4 are descriptive only and are
not reported here.

In [14]:
rows = []
for cr in CRIMES:
    y, d, X, names, state = load_crime(cr)
    D, _, _ = make_time_dummies(names, X, state); G = np.column_stack([np.ones(len(y)), D[:, 1:]])
    th, se, ns = pds_lasso_bch(y, d, X, G, state)       # BCH spec: time dummies + clustered SE
    bt, bse = BCH_T2["Post-double-selection"][cr]
    row = {"crime": cr, "BCH theta": bt, "BCH se": bse,
           "PDS-LASSO theta": round(th, 4), "PDS-LASSO se": round(se, 4), "controls": ns}
    if "cf" in main[cr]:
        row["SR-PDS-CF theta"] = round(main[cr]["cf"]["theta"], 4)
        row["SR-PDS-CF se"] = round(main[cr]["cf"]["se"], 4)
    rows.append(row)
display(pd.DataFrame(rows).set_index("crime"))
print("Valid inference throughout (HC1-free: clustered / cross-fit). In-sample SR-PDS SEs ($4) are descriptive only.")

,BCH theta,BCH se,PDS-LASSO theta,PDS-LASSO se,controls,SR-PDS-CF theta,SR-PDS-CF se
crime,,,,,,,
viol,-0.104,0.107,-0.1457,0.1116,5,-0.1709,0.1144
prop,-0.030,0.055,-0.0527,0.0550,9,-0.1015,0.0512
murd,-0.125,0.151,-0.1684,0.1969,8,-0.1795,0.1734


Valid inference throughout (HC1-free: clustered / cross-fit). In-sample SR-PDS SEs ($4) are descriptive only.


## 6. Summary and how to read these results

**Inference.** Report the cross-fitted, clustered SR-PDS-CF estimates as the headline. The in-sample
SR-PDS standard errors are *descriptive only*: the basis is fitted to the same data used for inference,
so those SEs are not valid and are labelled as such in $4. PDS-LASSO (HC1) and SR-PDS-CF (cross-fit +
cluster) are the defensible inference rows.

**What Levitt shows.** SR-PDS and SR-PDS-CF land essentially where PDS-LASSO and BCH land across all
three crimes. This is a *safety / validation* result, not a superiority result: on expert-curated data
where the hand-built dictionary already captures the structure, automating the basis does not change
the conclusion, and -- importantly -- SR does not manufacture an effect that is not there. The
superiority claim (SR-PDS beating DML-NN/RF/LASSO under genuine nonlinear confounding) is carried by
the simulations, not by Levitt.

**Replication ($5.1).** First-difference reproduces BCH exactly; Post-double-selection matches once the
cluster-robust penalty, time dummies and clustered SEs are in place.

**Recovery ($5.2-5.4).** Stripping the dictionary to primitives removes the hand-built nonlinear
controls; SR-PDS rediscovers nonlinear structure from primitives and, for murder and violent crime,
recovers the full-dictionary estimate. Watch the `recovers (<=1 SE)` flag in $5.3: if **property**
still overshoots after the degree cap, that is either a real nonlinearity worth reporting or a method
limitation -- decide which before writing it up, do not bury it. $5.4 shows which nonlinear terms load
on the treatment (confounder) versus outcome equation, which is the interpretability contribution.

**Caveat to state plainly.** The recovery experiment strips *BCH's own* dictionary and asks SR to
rebuild it -- a proof of concept that automation can match expertise where the expert answer already
exists. It does not, on its own, establish that automation helps where no expert basis is available;
that is the simulations' job.

**Discovery controls.** `MAX_DEGREE` caps total polynomial degree (default 4, matching the order of
BCH's dictionary) to prevent the high-order overfitting terms unconstrained SR produces; `POLY_ONLY`
additionally drops division for a purely polynomial basis. Constraining degree upstream and
cross-fitting downstream are complementary safeguards against overfitting.